In [363]:
import scvi
import torch
import numpy as np
from pathlib import Path
import pandas as pd
from modules.utils import dict_summary, capture_kwargs
import inspect
from typing import Literal
import scipy


In [2]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78521)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 64631)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 56563)

# #### Device() ####
# device = cuda:6

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [3]:
cortex = scvi.data.cortex(dataset_dir / 'scvi' / 'cortex')

INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin already downloaded   
INFO     Loading Cortex data from /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin  
INFO     Finished loading Cortex data                                                                              


/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [4]:
pbmc = scvi.data.pbmc_dataset(dataset_dir / 'scvi' / 'pbmc_dataset')

INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/gene_info_pbmc.csv already    
         downloaded                                                                                                
INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/pbmc_metadata.pickle already  
         downloaded                                                                                                
INFO     File                                                                                                      
         /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/pbmc8k/filtered_gene_bc_matrices.ta
         r.gz already downloaded                                                                                   
INFO     Extracting tar file                                                                                       
INFO     Removing extracted data at                                     

/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/site-packages/scvi/data/_built_in_data/_pbmc.py:75: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = pbmc8k.concatenate(pbmc4k)


In [5]:
breast = scvi.data.breast_cancer_dataset(dataset_dir / 'scvi' / 'breast_cancer_dataset')

INFO     File                                                                                                      
         /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/breast_cancer_dataset/Layer2_BC_count_matrix-1.t
         sv already downloaded                                                                                     
INFO     Loading dataset from                                                                                      
         /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/breast_cancer_dataset/Layer2_BC_count_matrix-1.t
         sv                                                                                                        
INFO     Finished loading dataset                                                                                  


In [6]:
cortex

AnnData object with n_obs × n_vars = 3005 × 19972
    obs: 'labels', 'precise_labels', 'cell_type'

In [7]:
pbmc

AnnData object with n_obs × n_vars = 11990 × 3346
    obs: 'n_counts', 'batch', 'labels', 'str_labels'
    var: 'gene_symbols', 'n_counts-0', 'n_counts-1', 'n_counts'
    uns: 'cell_types'
    obsm: 'design', 'raw_qc', 'normalized_qc', 'qc_pc'

In [8]:
pbmc

AnnData object with n_obs × n_vars = 11990 × 3346
    obs: 'n_counts', 'batch', 'labels', 'str_labels'
    var: 'gene_symbols', 'n_counts-0', 'n_counts-1', 'n_counts'
    uns: 'cell_types'
    obsm: 'design', 'raw_qc', 'normalized_qc', 'qc_pc'

In [9]:
pbmc

AnnData object with n_obs × n_vars = 11990 × 3346
    obs: 'n_counts', 'batch', 'labels', 'str_labels'
    var: 'gene_symbols', 'n_counts-0', 'n_counts-1', 'n_counts'
    uns: 'cell_types'
    obsm: 'design', 'raw_qc', 'normalized_qc', 'qc_pc'

In [10]:
torch.tensor(cortex.X)

tensor([[0, 3, 3,  ..., 0, 0, 0],
        [0, 1, 1,  ..., 0, 0, 0],
        [0, 0, 6,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [1, 1, 0,  ..., 0, 0, 0]], device='cuda:6', dtype=torch.int32)

In [11]:
torch.tensor(pbmc.X.toarray())

tensor([[0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [2., 2., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:6')

In [12]:
cortex.__dict__

{'_is_view': False,
 '_adata_ref': None,
 '_oidx': None,
 '_vidx': None,
 'file': Backing file manager: no file is set.,
 '_X': array([[0, 3, 3, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        [0, 0, 6, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [1, 1, 0, ..., 0, 0, 0]], shape=(3005, 19972), dtype=int32),
 '_obs':       labels  precise_labels          cell_type
 0          2               0       interneurons
 1          2               0       interneurons
 2          2               0       interneurons
 3          2               0       interneurons
 4          2               0       interneurons
 ...      ...             ...                ...
 3000       1               8  endothelial-mural
 3001       1               8  endothelial-mural
 3002       1               8  endothelial-mural
 3003       1               8  endothelial-mural
 3004       1               8  endothelial-mural
 
 [3005 rows x 3 columns],
 '

In [54]:
cortex.X.shape

(3005, 19972)

In [50]:
len(cortex.var.index.to_list())

19972

* TCGA Output:
    * x: counts tensor in (batch, genes)
    * x_labels: list of gene names in len(genes)
    * y: label tensor (int) in (batch,)
    * y_labels: list of class names in len(classes,)

* scVI output:
    * var.index.to_list(): x_labels (unfiltered)

provided:
    * int array 'X' in shape (samples, genes)
    * pd index 'names' with len (genes)
    * df 'name2ensg' with str cols ['ensg', 'name']
    * optional list 'ensg_filter'

filter out 'names' which do not have a corresponding name2ensg['name'] (non-case sensitive) in name2ensg['ensg'], 
and optionally, do not have a name2ensg['ensg'] in ensg_filter if provided.



In [41]:
brca.y_labels

['Basal', 'Her2', 'LumA', 'LumB', 'Solid Tissue Normal']

In [38]:
len(brca.x_labels)

4373

In [61]:
kegg.ensg

['ENSG00000000938',
 'ENSG00000001084',
 'ENSG00000001167',
 'ENSG00000001617',
 'ENSG00000001626',
 'ENSG00000001630',
 'ENSG00000001631',
 'ENSG00000002330',
 'ENSG00000002549',
 'ENSG00000002726',
 'ENSG00000003056',
 'ENSG00000003137',
 'ENSG00000003393',
 'ENSG00000003400',
 'ENSG00000003402',
 'ENSG00000003987',
 'ENSG00000004139',
 'ENSG00000004455',
 'ENSG00000004468',
 'ENSG00000004487',
 'ENSG00000004660',
 'ENSG00000004779',
 'ENSG00000004799',
 'ENSG00000004897',
 'ENSG00000004961',
 'ENSG00000004975',
 'ENSG00000005007',
 'ENSG00000005022',
 'ENSG00000005100',
 'ENSG00000005187',
 'ENSG00000005249',
 'ENSG00000005339',
 'ENSG00000005483',
 'ENSG00000005801',
 'ENSG00000005844',
 'ENSG00000005882',
 'ENSG00000005884',
 'ENSG00000005893',
 'ENSG00000005961',
 'ENSG00000006062',
 'ENSG00000006071',
 'ENSG00000006125',
 'ENSG00000006210',
 'ENSG00000006283',
 'ENSG00000006327',
 'ENSG00000006432',
 'ENSG00000006451',
 'ENSG00000006530',
 'ENSG00000006534',
 'ENSG00000006607',


In [48]:
brca.ensgv

,ensgv,ensg,name
0,ENSG00000000003.15,ENSG00000000003,TSPAN6
1,ENSG00000000005.6,ENSG00000000005,TNMD
2,ENSG00000000419.13,ENSG00000000419,DPM1
3,ENSG00000000457.14,ENSG00000000457,SCYL3
4,ENSG00000000460.17,ENSG00000000460,C1orf112
...,...,...,...
60655,ENSG00000288669.1,ENSG00000288669,NaN
60656,ENSG00000288670.1,ENSG00000288670,NaN
60657,ENSG00000288671.1,ENSG00000288671,NaN
60658,ENSG00000288674.1,ENSG00000288674,NaN


In [26]:
cortex.X

array([[0, 3, 3, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       [0, 0, 6, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0]], shape=(3005, 19972), dtype=int32)

In [30]:
cortex.obs['labels'].unique()

array([2, 6, 5, 4, 3, 1, 0])

In [33]:
cortex.obs['precise_labels'].unique()

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [35]:
cortex.obs['cell_type'].unique()

array(['interneurons', 'pyramidal SS', 'pyramidal CA1',
       'oligodendrocytes', 'microglia', 'endothelial-mural',
       'astrocytes_ependymal'], dtype=object)

In [60]:
cortex.var.index.to_series()

Tspan12              Tspan12
Tshz1                  Tshz1
Fnbp1l                Fnbp1l
Adamts15            Adamts15
Cldn12                Cldn12
                    ...     
Gm20738_loc4    Gm20738_loc4
Gm20738_loc6    Gm20738_loc6
Gm21943_loc1    Gm21943_loc1
Gm21943_loc3    Gm21943_loc3
Gm20738_loc3    Gm20738_loc3
Length: 19972, dtype: object

In [21]:
name2ensg = pd.read_csv(dataset_dir/'other'/'name2ensg.csv')

In [23]:
names = list(cortex.var.index)
names

['Tspan12',
 'Tshz1',
 'Fnbp1l',
 'Adamts15',
 'Cldn12',
 'Rxfp1',
 '2310042E22Rik',
 'Sema3c',
 'Jam2',
 'Apbb1ip',
 'Frem2',
 'BC005764',
 'Deptor',
 'C130030K03Rik',
 'Klhl13',
 'Tnfaip8l3',
 'Ascl1',
 'Atp1b2',
 'Tmem132e',
 'Prkar2b',
 'Necab1',
 'Nr2f2',
 'Stmn1-rs1',
 'Shisa9',
 'Sub1',
 'Vat1l',
 'Pcdh18',
 'Dpysl5',
 'Agtr2',
 'Cxcl14',
 'Ptprf',
 'Cers5',
 'Nacc2',
 'Cntnap2',
 'Asic4',
 'Tph2',
 'Rit2',
 'Scg2',
 'Nr2e1',
 'Bmp3',
 'B3galnt1',
 'Gm9846',
 'Kcnn1',
 'Fam19a2',
 'Samd4',
 'Stmn1',
 'Egln3',
 'Bmper',
 'Cd200',
 '1810041L15Rik',
 'Rnf128',
 'P2ry1',
 'Sorcs1',
 'Htr3a',
 'Kcnip1',
 'Sco2',
 '2610044O15Rik8',
 'Tmem41b',
 'Cdh9',
 'Ppp2r3d',
 'Usp28',
 'Trib2',
 'L2hgdh',
 'Cited2',
 'Btbd11',
 'Rgs12',
 'Cnr1',
 'Kcnd3',
 'Htr3b',
 'Sulf2',
 'Capn5',
 'Osbpl5',
 'Nrtn',
 'Nr2f1',
 'Galnt14',
 'Slc35f1',
 'Mmgt1',
 'Htr7',
 'E130309F12Rik',
 'Wtap',
 'Racgap1',
 'Arhgap24',
 'Maml3',
 'Gng2',
 'Tnc',
 'Fndc3a',
 'Npas3',
 'Celf6',
 'Inpp4b',
 'Npas1',
 'Ache',
 

In [22]:
name2ensg

,ensg,name
0,ENSG00000004059,ARF5
1,ENSG00000003056,M6PR
2,ENSG00000004478,FKBP4
3,ENSG00000003137,CYP26B1
4,ENSG00000003509,NDUFAF7
...,...,...
19380,ENSG00000132746,ALDH3B2
19381,ENSG00000196990,FAM163B
19382,ENSG00000188039,NWD1
19383,ENSG00000198728,LDB1


In [55]:
name2ensg

,ensg,name
0,ENSG00000004059,ARF5
1,ENSG00000003056,M6PR
2,ENSG00000004478,FKBP4
3,ENSG00000003137,CYP26B1
4,ENSG00000003509,NDUFAF7
...,...,...
19380,ENSG00000132746,ALDH3B2
19381,ENSG00000196990,FAM163B
19382,ENSG00000188039,NWD1
19383,ENSG00000198728,LDB1


In [15]:
kegg.relation

,set_idx,node_i_idx,node_j_idx,activation,binding/association,compound,dephosphorylation,dissociation,expression,glycosylation,indirect,indirect effect,inhibition,methylation,missing interaction,phosphorylation,repression,state change,ubiquitination
0,0,4131.0,314.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,0,4131.0,2201.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
2,0,4131.0,2821.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
3,0,3178.0,314.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,0,3178.0,2201.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75934,304,2013.0,3586.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75935,304,2013.0,1028.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75936,304,4266.0,2105.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
75937,304,4266.0,3586.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


In [67]:
import numpy as np
import pandas as pd
from typing import Iterable, Optional, Tuple

def filter_genes_by_ensg(
    X: np.ndarray,
    names: pd.Index,
    name2ensg: pd.DataFrame,
    ensg_filter: Optional[Iterable[str]] = None,
) -> Tuple[np.ndarray, pd.Index, pd.Series]:
    """
    Filter genes based on name↔ENSG mapping (case-insensitive), with optional ENSG filtering.

    Returns
    -------
    X_filt : np.ndarray
        Filtered expression matrix, shape (samples, kept_genes)
    names_filt : pd.Index
        Filtered gene names (original casing preserved)
    ensg_filt : pd.Series
        ENSG IDs aligned to names_filt
    """

    # --- normalize case ---
    names_lc = names.str.lower()

    n2e = name2ensg.copy()
    n2e["name_lc"] = n2e["name"].str.lower()

    # --- optional ENSG restriction ---
    if ensg_filter is not None:
        ensg_filter = set(map(str, ensg_filter))
        n2e = n2e[n2e["ensg"].isin(ensg_filter)]

    # --- build name → ENSG mapping ---
    name_to_ensg = (
        n2e
        .drop_duplicates("name_lc")
        .set_index("name_lc")["ensg"]
    )

    # --- find valid genes ---
    mask = names_lc.isin(name_to_ensg.index)

    # --- apply filtering ---
    X_filt = X[:, mask]
    names_filt = names[mask]
    ensg_filt = names_lc[mask].map(name_to_ensg)

    return X_filt, names_filt, ensg_filt


In [76]:
kegg.ensg

['ENSG00000000938',
 'ENSG00000001084',
 'ENSG00000001167',
 'ENSG00000001617',
 'ENSG00000001626',
 'ENSG00000001630',
 'ENSG00000001631',
 'ENSG00000002330',
 'ENSG00000002549',
 'ENSG00000002726',
 'ENSG00000003056',
 'ENSG00000003137',
 'ENSG00000003393',
 'ENSG00000003400',
 'ENSG00000003402',
 'ENSG00000003987',
 'ENSG00000004139',
 'ENSG00000004455',
 'ENSG00000004468',
 'ENSG00000004487',
 'ENSG00000004660',
 'ENSG00000004779',
 'ENSG00000004799',
 'ENSG00000004897',
 'ENSG00000004961',
 'ENSG00000004975',
 'ENSG00000005007',
 'ENSG00000005022',
 'ENSG00000005100',
 'ENSG00000005187',
 'ENSG00000005249',
 'ENSG00000005339',
 'ENSG00000005483',
 'ENSG00000005801',
 'ENSG00000005844',
 'ENSG00000005882',
 'ENSG00000005884',
 'ENSG00000005893',
 'ENSG00000005961',
 'ENSG00000006062',
 'ENSG00000006071',
 'ENSG00000006125',
 'ENSG00000006210',
 'ENSG00000006283',
 'ENSG00000006327',
 'ENSG00000006432',
 'ENSG00000006451',
 'ENSG00000006530',
 'ENSG00000006534',
 'ENSG00000006607',


In [77]:
out = filter_genes_by_ensg(
    X = cortex.X,
    names = cortex.var.index,
    name2ensg = name2ensg,
    ensg_filter = kegg.ensg,

)

In [78]:
cortex.X.shape

(3005, 19972)

In [79]:
out[0].shape

(3005, 3692)

In [80]:
out[1].shape

(3692,)

In [81]:
out[2].shape

(3692,)

In [82]:
cortex.obs

,labels,precise_labels,cell_type
0,2,0,interneurons
1,2,0,interneurons
2,2,0,interneurons
3,2,0,interneurons
4,2,0,interneurons
...,...,...,...
3000,1,8,endothelial-mural
3001,1,8,endothelial-mural
3002,1,8,endothelial-mural
3003,1,8,endothelial-mural


In [85]:
cortex.obs[['labels','cell_type']].drop_duplicates()

,labels,cell_type
0,2,interneurons
290,6,pyramidal SS
468,5,pyramidal CA1
1628,4,oligodendrocytes
2448,3,microglia
2546,1,endothelial-mural
2721,0,astrocytes_ependymal


In [89]:
pbmc.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 5489147 stored elements and shape (11990, 3346)>

In [112]:
pbmc.obs

,n_counts,batch,labels,str_labels
AAACCTGAGCTAGTGG-1,4520.0,0,2,CD4 T cells
AAACCTGCACATTAGC-1,2788.0,0,2,CD4 T cells
AAACCTGCACTGTTAG-1,4667.0,0,1,CD14+ Monocytes
AAACCTGCATAGTAAG-1,4440.0,0,1,CD14+ Monocytes
AAACCTGCATGAACCT-1,3224.0,0,3,CD8 T cells
...,...,...,...,...
TTTGGTTTCGCTAGCG-1,6514.0,1,1,CD14+ Monocytes
TTTGTCACACTTAACG-1,3293.0,1,3,CD8 T cells
TTTGTCACAGGTCCAC-1,8322.0,1,7,NK cells
TTTGTCAGTTAAGACA-1,2933.0,1,0,B cells


In [113]:
pbmc.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 5489147 stored elements and shape (11990, 3346)>

In [93]:
pbmc.var

,gene_symbols,n_counts-0,n_counts-1,n_counts
ENSG00000188976,NOC2L,1477.0,718.0,2135.0
ENSG00000187608,ISG15,4014.0,1888.0,5705.0
ENSG00000149527,PLCH2,121.0,41.0,157.0
ENSG00000157881,PANK4,224.0,107.0,326.0
ENSG00000157873,TNFRSF14,3639.0,1683.0,5193.0
...,...,...,...,...
ENSG00000183255,PTTG1IP,1483.0,711.0,2127.0
ENSG00000160255,ITGB2,18398.0,8663.0,26353.0
ENSG00000160299,PCNT,312.0,157.0,464.0
ENSG00000160305,DIP2A,832.0,400.0,1191.0


In [91]:
pbmc.obs[pbmc.obs['batch']==1]

,n_counts,batch,labels,str_labels
AAATGCCTCACCAGGC-1,4035.0,1,0,B cells
ACCCACTTCAGTCAGT-1,2461.0,1,7,NK cells
ACGGCCACAGGGTTAG-1,3618.0,1,1,CD14+ Monocytes
ACTGAGTGTGACCAAG-1,2735.0,1,0,B cells
ACTGATGCAGTCCTTC-1,4304.0,1,1,CD14+ Monocytes
...,...,...,...,...
TTTGGTTTCGCTAGCG-1,6514.0,1,1,CD14+ Monocytes
TTTGTCACACTTAACG-1,3293.0,1,3,CD8 T cells
TTTGTCACAGGTCCAC-1,8322.0,1,7,NK cells
TTTGTCAGTTAAGACA-1,2933.0,1,0,B cells


In [86]:
pbmc.obs[['labels','cell_type']].drop_duplicates()

KeyError: "['cell_type'] not in index"

In [ ]:
# brca = d.TCGA(
#     tcga_project = 'BRCA',
#     tcga_dir = dataset_dir/'tcga',
#     # type_col = 'sample_type',
#     subtype_col = 'paper_BRCA_Subtype_PAM50',
#     drop = ['Normal', 'Primary Tumor', 'Metastatic'],
#     gene_name_path = dataset_dir/'other'/'name2ensg.csv',
#     keep_noname = False,
# )

In [119]:
cortex.obs['cell_type'].value_counts()

cell_type
pyramidal CA1           939
oligodendrocytes        820
pyramidal SS            399
interneurons            290
endothelial-mural       235
astrocytes_ependymal    224
microglia                98
Name: count, dtype: int64

In [122]:
pbmc.obs['str_labels'].value_counts()

str_labels
CD4 T cells          4996
CD14+ Monocytes      2227
B cells              1621
CD8 T cells          1448
Other                 463
NK cells              457
FCGR3A+ Monocytes     351
Dendritic Cells       339
Megakaryocytes         88
Name: count, dtype: int64

In [ ]:
class scviData:
    def __init__(
        self,
        dataset: str,
        dataset_dir: Path | str,

        x_label_col: str,
        x_label_type: Literal['ensg','name'],
        gene_name_path: str|Path,
        y_label_col: str = 'labels',

        keep_noname: bool = False,
        verbose: bool = True
    ):

        # get kwargs
        sig = inspect.signature(type(self).__init__)
        self._orig_kwargs = capture_kwargs(sig, self, dataset, dataset_dir, x_label_col, x_label_type, gene_name_path, y_label_col, keep_noname, verbose)
        self.verbose = verbose

        dataset_dir = Path(dataset_dir)
        self.adata = self._load_adata(dataset, dataset_dir/dataset)
        self.x_names, self.ensg_complete = self._get_x_names(x_label_col, x_label_type, gene_name_path, keep_noname)
        self.x_labels = self.x_names['name'].to_list()
        self.y, self.y_labels = self._get_labels(y_label_col)

    def __str__(self):
        return dict_summary(self.__dict__)

    def _load_adata(self, dataset:str, dataset_dir:str):
        try:
            fn = getattr(scvi.data, dataset)
        except AttributeError:
            raise AttributeError(f"{dataset} not in scvi.data")

        if not callable(fn):
            raise TypeError(f"{dataset} exists but is not callable")

        return fn(dataset_dir)

    def _get_x_names(self, label_col:str, label_type:str, gene_name_path:str|Path, keep_noname:bool):
        # data
        df:pd.DataFrame = self.adata.var.reset_index().copy()
        ref = pd.read_csv(gene_name_path)

        # normalize input
        if label_type == 'name':
            df[label_col] = df[label_col].astype('string').str.lower()
        ref['ensg_key'] = ref['ensg']#.astype('string').str.lower()
        ref['name_key'] = ref['name'].astype('string').str.lower()

        # complete name table
        if label_type == 'ensg':
            ensg2name = (
                pd.Series(ref["name"].to_numpy(), index=ref["ensg_key"])
                .groupby(level=0)
                .last()
            )
            df["ensg"] = df[label_col].astype('string')
            df["name"] = df[label_col].map(ensg2name)

        else:  # x_label_type == 'name'
            name2ensg = (
                pd.Series(ref["ensg"].to_numpy(), index=ref["name_key"])
                .groupby(level=0)
                .last()
            )
            df["name"] = df[label_col].astype('string')
            df["ensg"] = df[label_col].map(name2ensg)

        # get complete list of ensg
        ensg_complete = df['ensg'].drop_duplicates().tolist() if keep_noname else df['ensg'][~df['name'].isna()].drop_duplicates().tolist()

        return df, ensg_complete
    
    def _get_labels(self, y_label_col:str):
        cats = pd.Categorical(self.adata.obs[y_label_col])
        y = torch.tensor(cats.codes)
        y_labels = list(cats.categories)
        return y, y_labels

    def get_counts(self, ensg_filter:list[int]|None=None):
        x = self.adata.X
        if scipy.sparse.issparse(x):
            x = x.toarray()

        # filter
        if ensg_filter is not None:
            keep = self.x_names['ensg'].astype("string").isin(pd.Index(ensg_filter))
            x = x[:, keep.to_numpy()]
            self.x_names = self.x_names.loc[keep]

        # convert to tensor in (b,n,1)
            
        self.x = torch.tensor(x).unsqueeze(-1)

        if self.verbose:
            print('# #### scviData() ####')
            print(dict_summary(self.__dict__))

In [330]:
scdata = scviData(
    dataset='cortex', # cortex, pbmc_dataset
    dataset_dir=dataset_dir / 'scvi',
    x_label_col='index',
    x_label_type='name', # name, ensg
    y_label_col='cell_type',
    gene_name_path=dataset_dir/'other'/'name2ensg.csv',
)
scdata.get_counts()
scdata.x.shape

INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin already downloaded   
INFO     Loading Cortex data from /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin  
INFO     Finished loading Cortex data                                                                              


/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


torch.Size([3005, 19972])

In [332]:
scdata.y_labels

['astrocytes_ependymal',
 'endothelial-mural',
 'interneurons',
 'microglia',
 'oligodendrocytes',
 'pyramidal CA1',
 'pyramidal SS']

In [341]:
scdata.x_names

,index,name,ensg
2,fnbp1l,fnbp1l,ENSG00000137942
4,cldn12,cldn12,ENSG00000157224
7,sema3c,sema3c,ENSG00000075223
8,jam2,jam2,ENSG00000154721
9,apbb1ip,apbb1ip,ENSG00000077420
...,...,...,...
19905,rragb,rragb,ENSG00000083750
19919,rps6ka3,rps6ka3,ENSG00000177189
19925,phka2,phka2,ENSG00000044446
19936,ctps2,ctps2,ENSG00000047230


In [355]:
pbmc.obs

,n_counts,batch,labels,str_labels
AAACCTGAGCTAGTGG-1,4520.0,0,2,CD4 T cells
AAACCTGCACATTAGC-1,2788.0,0,2,CD4 T cells
AAACCTGCACTGTTAG-1,4667.0,0,1,CD14+ Monocytes
AAACCTGCATAGTAAG-1,4440.0,0,1,CD14+ Monocytes
AAACCTGCATGAACCT-1,3224.0,0,3,CD8 T cells
...,...,...,...,...
TTTGGTTTCGCTAGCG-1,6514.0,1,1,CD14+ Monocytes
TTTGTCACACTTAACG-1,3293.0,1,3,CD8 T cells
TTTGTCACAGGTCCAC-1,8322.0,1,7,NK cells
TTTGTCAGTTAAGACA-1,2933.0,1,0,B cells


In [366]:
type(pbmc.X.toarray())

numpy.ndarray

In [362]:
type(cortex.X)

numpy.ndarray

In [361]:
type(pbmc.X)

scipy.sparse._csr.csr_matrix

In [368]:
scdata = scviData(
    dataset='cortex', # cortex, pbmc_dataset
    dataset_dir=dataset_dir / 'scvi',
    x_label_col='index',
    x_label_type='name', # name, ensg
    y_label_col='cell_type',
    gene_name_path=dataset_dir/'other'/'name2ensg.csv',
)

kegg2 = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=scdata,
)

dataset2 = d.GraphDataset(scdata, kegg2, kegg2)
_batch = d.get_toy_databatch(dataset, device)

INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin already downloaded   
INFO     Loading Cortex data from /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/cortex/expression.bin  
INFO     Finished loading Cortex data                                                                              


/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     3688                     list
# pathway_labels           305                      list
# edge_index               (2, 26946)               Tensor (cuda:6)
# edge_attr                (26946, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (3688, 305)              Tensor (cuda:6)

# #### scviData() ####
# _orig_kwargs             8                        dict
# verbose                  True                     bool
# adata                    AnnData
# x_names                  (3688, 3)                DataFrame
# ensg_complete            14404                    list
# x_labels                 19972                    list
# y                        (3005,)                  Tensor (cuda:6)
# y_labels                 7                        list
# x          

In [369]:
scdata = scviData(
    dataset='pbmc_dataset', # cortex, pbmc_dataset
    dataset_dir=dataset_dir / 'scvi',
    x_label_col='index',
    x_label_type='ensg', # name, ensg
    y_label_col='str_labels', # cell_type, str_labels
    gene_name_path=dataset_dir/'other'/'name2ensg.csv',
)

kegg2 = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=scdata,
)

dataset2 = d.GraphDataset(scdata, kegg2, kegg2)
_batch = d.get_toy_databatch(dataset, device)

INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/gene_info_pbmc.csv already    
         downloaded                                                                                                
INFO     File /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/pbmc_metadata.pickle already  
         downloaded                                                                                                
INFO     File                                                                                                      
         /home/mv18gs/Documents/GitHub/pathway_model/datasets/scvi/pbmc_dataset/pbmc8k/filtered_gene_bc_matrices.ta
         r.gz already downloaded                                                                                   
INFO     Extracting tar file                                                                                       
INFO     Removing extracted data at                                     

/home/mv18gs/miniconda3/envs/thesis_scvi/lib/python3.14/site-packages/scvi/data/_built_in_data/_pbmc.py:75: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = pbmc8k.concatenate(pbmc4k)


# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     1098                     list
# pathway_labels           305                      list
# edge_index               (2, 2117)                Tensor (cuda:6)
# edge_attr                (2117, 16)               Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (1098, 305)              Tensor (cuda:6)

# #### scviData() ####
# _orig_kwargs             8                        dict
# verbose                  True                     bool
# adata                    AnnData
# x_names                  (1098, 7)                DataFrame
# ensg_complete            3337                     list
# x_labels                 3346                     list
# y                        (11990,)                 Tensor (cuda:6)
# y_labels                 9                        list
# x          